# hunches walkthrough

The detailed tour of the chain after you have candidates: estimate the cost, classify with a real
model under a spend cap, evaluate against labels people made, read the disagreements, tune the prompt
on dev, price a larger run, evaluate the test split once, and keep the classifier.

It runs over the one corpus every example in `e2e/` shares (`../common/`: 60 short workplace posts
with hand-built 2-D vectors, not real embeddings). The other examples run these steps tersely against
a file, Postgres and S3; this is the one that explains each.

**What needs you:**

- `ANTHROPIC_API_KEY` in the environment for the first classification (nothing here reads or writes a
  key file). Everything before the estimate runs offline.
- Gold labels made by **people** in the labelling page (`hn.label`). This notebook uses the file they
  committed, `../common/gold.jsonl`, copied byte for byte and never edited. An agent working with you
  must not write or fill in those labels.

Results are cached in `../.cache/`, shared with the other examples, so rerunning costs nothing.

In [ ]:
import json
import os
import shutil
import tempfile
from pathlib import Path

from pydantic_ai import Embedder
from pydantic_ai.embeddings import EmbeddingResult, TestEmbeddingModel
from pydantic_ai.usage import RequestUsage

import hunches as hn

COMMON = (
    Path("..") / "common"
)  # the corpus, taxonomy, prompt, seeds and gold every example shares
CACHE = Path("..") / ".cache"  # model results, shared by every example, git-ignored
work = Path(
    tempfile.mkdtemp(prefix="hunches-walkthrough-")
)  # working files stay out of the repo
hn.settings.root = work
work

In [ ]:
SEED_VECTORS = json.loads((COMMON / "seeds.json").read_text())


class HandEmbedder(Embedder):
    """Stand-in for a real embedding model: a seed phrase maps to its hand-built vector.

    With your own corpus, drop this and pass `embedder="openai:text-embedding-3-small"` (or the
    model your corpus was embedded with) to `hn.get_candidates`.
    """

    def __init__(self):
        super().__init__(TestEmbeddingModel("hand-built-2d"))

    async def embed_query(self, query, *, settings=None):
        return EmbeddingResult(
            [SEED_VECTORS[query]],
            inputs=[query],
            input_type="query",
            model_name="hand-built-2d",
            provider_name="test",
            usage=RequestUsage(input_tokens=1),
        )

## 1. Candidates

`get_candidates` takes a corpus and returns the items to classify. With seed phrases it is a semantic
search: it embeds them, keeps the items closest to any seed, and each item keeps its best similarity.
The result is saved to `candidates.jsonl` in the working folder.

In [ ]:
corpus = hn.Table(
    COMMON / "corpus.jsonl",
    id_column="id",
    vector_column="embedding",
    embedding_model="hand-built-2d",
)
seeds = list(SEED_VECTORS)
candidates = hn.get_candidates(corpus, seeds=seeds, embedder=HandEmbedder(), floor=0.65)
candidates

In [ ]:
candidates.to_polars()  # .to_pandas() works the same way

Seeds are one way in. Keywords or regular expressions, or simply the whole corpus, need no
embeddings at all. The whole corpus is what this walkthrough classifies below.

In [ ]:
keyword_hits = hn.get_candidates(
    hn.Table(COMMON / "corpus.jsonl", id_column="id"),
    keywords=["laid off", "redundan"],
    out=False,
)
keyword_hits.to_polars().head()

In [ ]:
everything = hn.get_candidates(
    hn.Table(COMMON / "corpus.jsonl", id_column="id"), out=False
)
everything

## 2. Labels and a first prompt

`single` means one label per post; `off_topic` is built in. The classifier is immutable: a new prompt
is a new classifier (`with_prompt`), so you can keep both and compare them.

In [ ]:
taxonomy = hn.Taxonomy.read(COMMON / "taxonomy.yaml")
prompt = (COMMON / "prompt.md").read_text().strip()
clf = hn.Classifier("anthropic:claude-haiku-5-5", taxonomy, prompt, cache=CACHE)
clf

## 3. Gold: people label

`gold.draw` picks pending rows with no labels. `hn.label(gold, taxonomy)` opens the labelling page in a
browser, where a person decides every label; the page never shows the classifier's guess. For this
repository a maintainer did that once and committed the result, so the notebook copies the file
unchanged. (Drawing, shown on a scratch file below, never writes a label.)

In [ ]:
scratch = hn.Gold(work / "scratch.jsonl")
scratch.draw(candidates, n=3, split="dev", seed=1)

In [ ]:
if not (COMMON / "gold.jsonl").exists():
    raise RuntimeError(
        "SKIPPED: e2e/common/gold.jsonl is not made yet: a person labels it, see e2e/README.md"
    )
shutil.copyfile(COMMON / "gold.jsonl", work / "gold.jsonl")
print((COMMON / "GOLD.md").read_text())
gold = hn.Gold(work / "gold.jsonl")
gold

## 4. Estimate before you spend

`estimate` calls no model: it counts what the cache already has and prices the rest from recorded
usage. An unknown price shows as `?`, never as `$0`. The cell after it stops here when there is no key
and something is not cached.

In [ ]:
estimate = clf.estimate(everything)
estimate

In [ ]:
if estimate.live and not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        f"SKIPPED: set ANTHROPIC_API_KEY ({estimate.live} items are not cached)"
    )

In [ ]:
# A price is only known after one priced call, so with none recorded one item is classified first.
if estimate.live and estimate.dollars is None:
    clf.classify(everything.rows[:1], progress=False)
    estimate = clf.estimate(everything)
estimate

## 5. Classify under a cap

`max_cost` refuses before spending anything if the estimate is over it or the price is unknown, and
stops mid-run if the spend crosses it. Rerunning this cell makes no calls (they are cached).

In [ ]:
results = clf.classify(everything, out=work / "results.jsonl", max_cost=0.50)
results

In [ ]:
results.to_polars()

## 6. Evaluate on the dev split

Every evaluation is appended to `evals.jsonl` in the working folder. The gold rows are corpus items,
so they are already classified above.

In [ ]:
ev = hn.evaluate(clf, gold.dev, target=0.7)
ev

In [ ]:
ev.disagreements_to_polars()  # where the classifier and the people differ, with its reasoning

## 7. Tune the prompt on dev, never on test

Change one thing, build a new classifier, evaluate again. Only the changed prompt costs calls.

In [ ]:
clf2 = clf.with_prompt(
    prompt + " A post about rumoured or expected redundancies counts as layoff."
)
ev2 = hn.evaluate(clf2, gold.dev, target=0.7)
ev2

## 8. The test split, once

Evaluate the test split once, at the end, to report the number. It must not guide any edit.

In [ ]:
final = hn.evaluate(clf2, gold.test, target=0.7)
final

In [ ]:
hn.eval_log()  # every evaluation so far, with how often the test split was used

## 9. Pick a cutoff, price the run, run it

`band_sample` classifies a few items per similarity band and shows how much of each is off topic. The
hand-built vectors put every candidate in the top band; with real embeddings the bands spread out.
You choose the cutoff.

In [ ]:
report = hn.band_sample(clf2, candidates, per_band=5)
report

In [ ]:
chosen = candidates.above(0.75)
clf2.estimate(chosen)

In [ ]:
chosen_results = clf2.classify(chosen, out=work / "chosen.jsonl", max_cost=0.50)
chosen_results.to_polars()

## 10. Keep the classifier

A saved bundle (model, taxonomy, prompt, the final evaluation) is plain files to commit. A pipeline
loads it with `hn.Classifier.load("classifiers/layoffs")`, with no gold and no UI.

In [ ]:
clf2.save(work / "classifiers" / "layoffs", evaluation=final)